# 27 — DOCX Parsing
**Goal:** Extract text and structure from .docx resumes.

After PDF (Ch. 26), the second most common resume format is the Word `.docx` — and it is far friendlier to parse. A `.docx` is a ZIP archive of XML files in which paragraphs, styles, tables, headers, and footers are explicit objects, not positioned glyphs. This chapter creates a resume in that format and reads its structure back with `python-docx`.

**Why it matters for resumes / ATS:** section headers and skill matrices are real, queryable objects in DOCX — a heading is a `Heading 1` paragraph, a skills matrix is a table. That structure maps almost one-to-one onto the sections an ATS needs to build (Ch. 32), which makes DOCX the highest-fidelity input path in the whole extraction pipeline.

## 1. Understanding DOCX Structure

Unzip a `.docx` and you find an XML package: `word/document.xml` holds the body content, `word/styles.xml` the formatting definitions, and `word/header*.xml` the running headers. `python-docx` wraps all of it so you work with `Document`, `Paragraph`, and `Table` objects instead of raw XML.

**What the code does:** prints that ZIP layout one line per part, then the punchline — DOCX is much easier than PDF because *sections, styles, and tables are explicit*: a heading knows it is a heading, a cell knows it is a cell.

**Expected output:** the three XML paths (`document.xml`, `styles.xml`, `header*.xml`), then the note that `python-docx` handles all of it. Compare with Ch. 26: nothing here needs coordinate math.

In [ ]:
print('''DOCX is a ZIP of XML files:\n- word/document.xml -> main content\n- word/styles.xml -> formatting\n- word/header*.xml -> headers\n\npython-docx handles all of this.\nDOCX is much easier than PDF because sections, styles, and tables are explicit.''')

## 2. Creating and Extracting DOCX

Round-trip time: the cell writes a small resume with `python-docx`, saves it, reopens the same file, and prints every non-empty paragraph with its **style name**. The style name is the key — it is stored in the XML and comes back intact, so `Title` and `Heading 1` paragraphs announce themselves.

**What the code does:**
- Sets the `Normal` style to Arial 11pt, then adds a `level=0` heading (the name), a contact line, `level=1` headings (`PROFESSIONAL SUMMARY`, `EXPERIENCE`) and body paragraphs
- Uses `add_run(...).bold = True` for the employer line — runs carry inline formatting inside a paragraph
- Saves to `/tmp/test_resume.docx` (about 37 KB in this environment — mostly XML boilerplate)
- Reopens and prints `[style] text` for each non-empty paragraph

**Expected:** `[Title] Srivatsa Gorti`, a `[Normal]` contact line, `[Heading 1] PROFESSIONAL SUMMARY`, the summary paragraph, `[Heading 1] EXPERIENCE`, and the job lines — the section skeleton of the resume recovered for free.

In [ ]:
from docx import Document
from docx.shared import Pt

doc = Document()
style = doc.styles['Normal']
style.font.name = 'Arial'
style.font.size = Pt(11)

doc.add_heading('Srivatsa Gorti', level=0)
doc.add_paragraph('srivatsa@email.com | +1-555-1234')
doc.add_heading('PROFESSIONAL SUMMARY', level=1)
doc.add_paragraph('Data scientist with 5+ years experience in Python, NLP, and ML.')
doc.add_heading('EXPERIENCE', level=1)
p = doc.add_paragraph()
p.add_run('Google, Mountain View').bold = True
p.add_run(' — Senior Data Scientist')
doc.add_paragraph('Jan 2020 - Present')
doc.add_paragraph('Developed NLP pipelines processing 10M+ documents daily')

test_docx = '/tmp/test_resume.docx'
doc.save(test_docx)

doc2 = Document(test_docx)
print("=== Extracted Content ===")
for para in doc2.paragraphs:
    if para.text.strip():
        print(f"  [{para.style.name:20s}] {para.text[:60]}")

## 3. Table Extraction

Skills matrices are the most table-shaped part of a resume, and DOCX stores them as real tables. `doc.tables` gives every table in the document; each one exposes `rows`, `columns`, and `row.cells` — no coordinate math required.

**What the code does:**
- Builds a 4×3 "Skills Matrix" table with the `Light Grid Accent 1` style, fills a header row (Skill / Level / Years) and three data rows
- Saves and reopens it, then iterates `doc4.tables`, printing each table's dimensions and every row's cell texts

**Expected:** `Table 1: 4 rows x 3 cols`, then the header row followed by the three skill rows (`Python`/`Expert`/`5`, `TensorFlow`/`Advanced`/`3`, `NLP`/`Advanced`/`4`). Rows-of-cells like this are ready to become structured skill lists for keyword matching — no parsing needed, just iteration.

In [ ]:
# Create a skills table
doc3 = Document()
doc3.add_heading('Skills Matrix', level=1)
table = doc3.add_table(rows=4, cols=3)
table.style = 'Light Grid Accent 1'
for i, (skill, level_, years) in enumerate([
    ("Python", "Expert", "5"), ("TensorFlow", "Advanced", "3"), ("NLP", "Advanced", "4"),
]):
    table.rows[i+1].cells[0].text = skill
    table.rows[i+1].cells[1].text = level_
    table.rows[i+1].cells[2].text = years
table.rows[0].cells[0].text = "Skill"
table.rows[0].cells[1].text = "Level"
table.rows[0].cells[2].text = "Years"
doc3.save("/tmp/skills.docx")

doc4 = Document("/tmp/skills.docx")
for i, table in enumerate(doc4.tables):
    print(f"Table {i+1}: {len(table.rows)} rows x {len(table.columns)} cols")
    for row in table.rows:
        print(f"  {[cell.text for cell in row.cells]}")

## Summary: DOCX preserves structure. python-docx handles paragraphs, styles, tables, headers, footers.

**DOCX gives you structure for free — so use it.** Where PDF parsing (Ch. 26) reconstructs paragraphs from coordinates, `python-docx` hands you `Paragraph`, `style.name`, and `Table` objects directly. The style names alone (`Title`, `Heading 1`) are a credible section map, which Ch. 32 will build on for section detection.

One caveat: not every resume reaches you as a `.docx`. Scanned pages and image-only files have no structure to preserve — those need OCR, which is exactly the fallback path in Ch. 28.